# Session 3 notebook: three lenses on one stock

**Question.** Micron (MU) is our running example from Exercise 1. Its stock earned a very large return after 2020. Did it beat what its characteristics and risks would predict? You will answer it three ways and compare:

| Lens | Idea | What is estimated |
|---|---|---|
| **DGTW** (Part 2) | Compare the stock with a benchmark portfolio of look-alike stocks | Nothing: stock return minus benchmark return |
| **Factor regression** (Part 3) | Regress the stock's returns on Mkt, SMB, HML, UMD | The stock's loadings (time series) |
| **Fundamental model** (Part 4) | Every month, regress *all* stocks on their exposures | The factor returns (cross section) |

**Before class (about 25 min): Parts 0-3.** Load the data, then get Micron's DGTW abnormal return and its Carhart alpha. These are mostly read-and-run. Write down the two numbers: you need them in Part 5.

**In class (about 45 min): Parts 4-5.** Build the fundamental model, attribute Micron's return, then compare the three lenses in groups.

**Files** (in the class folder; see Part 0): `class_panel.parquet`, `dgtw_stocks.parquet`, `ff_factors_monthly.parquet`, `ff48_names.csv`, `barra_factor_returns_ref.csv`.

Cells marked **Your turn** work as written. Change the value, rerun, and say what changed and why.

## Part 0. Setup (before class)

In [ ]:
import numpy as np, pandas as pd, statsmodels.api as sm
import matplotlib.pyplot as plt
from pathlib import Path
pd.set_option("display.width", 160, "display.max_columns", 30, "display.float_format", "{:,.4f}".format)

# The class data (data.zip). The link is on the course notes page, not in this public notebook.
DATA_ZIP_LINK = ""     # <- paste the Google Drive link to data.zip between the quotes, then run this cell

import re, zipfile, urllib.request
DATA_DIR = Path("data")
if not (DATA_DIR / "class_panel.parquet").exists():
    fid = re.search(r"/d/([\w-]+)|[?&]id=([\w-]+)", DATA_ZIP_LINK)
    assert fid, "Paste the data.zip link from the course notes page into DATA_ZIP_LINK above, then rerun this cell."
    fid = fid.group(1) or fid.group(2)
    urllib.request.urlretrieve(f"https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t", "data.zip")
    assert zipfile.is_zipfile("data.zip"), "The download did not return a zip. Check that the link is shared with you, and try again."
    zipfile.ZipFile("data.zip").extractall(".")          # creates ./data/
    print("downloaded and unzipped data.zip")

panel = pd.read_parquet(DATA_DIR / "class_panel.parquet")
dgtw  = pd.read_parquet(DATA_DIR / "dgtw_stocks.parquet")
ff    = pd.read_parquet(DATA_DIR / "ff_factors_monthly.parquet").set_index("month")
ind_names = pd.read_csv(DATA_DIR / "ff48_names.csv").set_index("industry")["industry_name"]
ref_factors = pd.read_csv(DATA_DIR / "barra_factor_returns_ref.csv", index_col=0)

STYLES = ["size", "value", "momentum", "beta", "resid_vol", "leverage", "liquidity", "profitability"]
print(f"panel: {len(panel):,} stock-months, {panel.permno.nunique():,} stocks, {panel.month.min()} to {panel.month.max()}")
print(f"dgtw : {len(dgtw):,} stock-months, {dgtw.month.min()} to {dgtw.month.max()}")

In [ ]:
def nw_mean(x, lags=3):
    """Average of a series and its Newey-West t-statistic (robust to autocorrelation)."""
    x = np.asarray(x, float)
    fit = sm.OLS(x, np.ones(len(x))).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    return fit.params[0], fit.tvalues[0]

# The stock to study: Micron, our running example. Afterward you can try any other ticker in the panel.
TICKER = "MU"
START, END = "2020-01", "2026-08"

permno = panel.loc[panel.ticker == TICKER, "permno"].mode().iloc[0]
print(TICKER, "-> permno", permno)

## Part 1. The data (before class)

The panel has one row per stock and month. `y` is the excess return (return minus the risk-free rate). `me_prev` is the market cap at the *end of the previous month*. The eight style columns are exposures that were known at the start of the month, already standardized: cap-weighted average 0, standard deviation 1.

In [ ]:
panel[panel.ticker == TICKER].tail(3)

In [ ]:
# One month, all stocks. This is the cross-section that Part 4 will regress.
m = panel[panel.month == "2024-06"]
print(len(m), "stocks in 2024-06")
m[STYLES].describe().loc[["mean", "std", "min", "max"]]

**Check yourself.** The plain mean of `size` is not zero here, but the cap-weighted mean is. Why would a model standardize with market-cap weights?

In [ ]:
w = m["me_prev"]
print("plain mean of size        :", round(m["size"].mean(), 3))
print("cap-weighted mean of size :", round(np.average(m["size"], weights=w), 3))

## Part 2. Lens 1: DGTW (before class)

Each stock is assigned to one of 125 benchmark portfolios (5 size x 5 book-to-market x 5 momentum) and compared with it. `port` is the three-digit code: size, value, momentum, each 1 to 5.

**Caution.** `dgtw_stocks.parquet` is the instructor's *replication* of DGTW from CRSP and Compustat. It matches the official WRDS table only loosely (correlation about 0.76, because the benchmark differs; stock returns match). The optional cell in 2.4 lets you swap in the official WRDS series.

In [ ]:
d = dgtw[(dgtw.permno == permno)].sort_values("date").set_index("month")
d = d.loc[START:END]
d["port_size"], d["port_bm"], d["port_mom"] = d["port"].astype(str).str[0], d["port"].astype(str).str[1], d["port"].astype(str).str[2]
print((d["ret"] - d["bench_ret"] - d["dgtw_xret"]).abs().max(), "<- dgtw_xret = ret - bench_ret, up to rounding")
d[["ret", "bench_ret", "dgtw_xret", "port", "port_size", "port_bm", "port_mom"]].head(8)

**Read the portfolio code.** A `port` of 541 means size quintile 5 (largest), book-to-market quintile 4, momentum quintile 1 (weakest prior-year return). Does your stock's code change over time? When and why? (Portfolios are re-formed each July.)

In [ ]:
dgtw_mean, dgtw_t = nw_mean(d["dgtw_xret"])
print(f"{TICKER}: average DGTW abnormal return {100*dgtw_mean:.2f}% per month, Newey-West t = {dgtw_t:.2f}, {len(d)} months")

fig, ax = plt.subplots(figsize=(7, 3.2))
(1 + d["ret"]).cumprod().plot(ax=ax, label=f"{TICKER}")
(1 + d["bench_ret"]).cumprod().plot(ax=ax, label="its DGTW benchmark")
ax.set_title(f"{TICKER} vs. its characteristic-matched benchmark (growth of $1)"); ax.legend(); ax.set_xlabel("")
plt.tight_layout(); plt.show()

### 2.4 Optional: use the official WRDS series (needs your WRDS login)
The official `wrdsapps.mastertable.dgtw_xret` ends in Dec 2024. Set `RUN_WRDS = True` to compare. Do not splice the official and replicated series together; use one or the other for a whole sample.

In [ ]:
RUN_WRDS = False
if RUN_WRDS:
    import wrds
    db = wrds.Connection()
    off = db.raw_sql(f"""SELECT permno, date, dgtw_xret AS official_xret FROM wrdsapps.mastertable
                       WHERE permno = {int(permno)} AND date >= '2020-01-01' AND dgtw_xret IS NOT NULL""", date_cols=["date"])
    db.close()
    off["month"] = off["date"].dt.to_period("M").astype(str)
    both = d.reset_index().merge(off[["month", "official_xret"]], on="month")
    print("months compared:", len(both), "| correlation:", round(both["dgtw_xret"].corr(both["official_xret"]), 2))
    print("official average: %.2f%% per month, t = %.2f" % ((lambda r: (100*r[0], r[1]))(nw_mean(both["official_xret"]))))

## Part 3. Lens 2: factor regression, Carhart (before class)

Same regression as Exercise 2. The stock's excess return on Mkt-RF, SMB, HML, UMD. The intercept is alpha. The loadings are *estimated* from the stock's own history.

In [ ]:
y = panel[(panel.permno == permno)].set_index("month")["y"].loc[START:END]
X = sm.add_constant(ff.loc[y.index, ["mktrf", "smb", "hml", "umd"]])
carhart = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 3})
print(carhart.summary().tables[1])
print(f"\nalpha = {100*carhart.params['const']:.2f}% per month (t = {carhart.tvalues['const']:.2f}),  R-squared = {carhart.rsquared:.2f}")

**Check yourself.** Which loadings are statistically significant, and which are not? What does a market loading of about 1.7 say about how Micron moves when the market moves 1%? Notice that the alpha's t-statistic is below 2 even though the alpha is large: why?

## Part 4. Lens 3: a fundamental (Barra-style) model (in class)

This is the idea from the lecture slides, run on real data. Each month, across all stocks:

> excess return of stock *i* = country + industry<sub>i</sub> + &Sigma;<sub>k</sub> exposure<sub>ik</sub> &times; style return<sub>k</sub> + specific<sub>i</sub>

The exposures are known. The *style returns* are the unknowns, estimated by weighted least squares with weights equal to the square root of market cap. The industry returns are constrained so their cap-weighted average is zero, otherwise they could not be told apart from the country factor. What is left over is the stock's **specific return**.

This is a simplified teaching model with 8 styles and 48 industries, not the commercial MSCI Barra model.

In [ ]:
def one_month(g):
    """Cross-sectional WLS for one month. Returns the factor returns (a Series) and a weighted R-squared."""
    inds = sorted(g["industry"].unique())
    cap = g.groupby("industry")["me_prev"].sum()
    share = cap / cap.sum()
    ref = share.idxmax()                            # drop the biggest industry; its return is implied by the constraint
    D = pd.get_dummies(g["industry"]).reindex(columns=inds, fill_value=0).astype(float)
    cols = {"country": np.ones(len(g))}
    for j in inds:
        if j != ref:
            cols[f"ind_{int(j)}"] = D[j].values - (share[j] / share[ref]) * D[ref].values
    for s in STYLES:
        cols[s] = g[s].values.astype(float)
    X = pd.DataFrame(cols, index=g.index)
    w = np.sqrt(g["me_prev"].values)               # WLS weight = sqrt(market cap)
    rw = np.sqrt(w)[:, None]
    beta, *_ = np.linalg.lstsq(X.values * rw, g["y"].values[:, None] * rw, rcond=None)
    f = pd.Series(beta.ravel(), index=X.columns)
    f[f"ind_{int(ref)}"] = -sum(share[j] / share[ref] * f[f"ind_{int(j)}"] for j in inds if j != ref)
    resid = g["y"].values - X.values @ beta.ravel()
    ybar = np.average(g["y"].values, weights=w)
    r2 = 1 - np.sum(w * resid**2) / np.sum(w * (g["y"].values - ybar) ** 2)
    return f, r2

MONTH = "2024-06"          # Your turn: try another month
f, r2 = one_month(panel[panel.month == MONTH])
print(f"{MONTH}: weighted R-squared = {r2:.2f}")
print("\nstyle returns (% per unit of exposure, i.e. per 1 std dev):")
print((100 * f[["country"] + STYLES]).round(2).to_string())

**Read it.** Each style return is what the market paid *that month* for one standard deviation of that characteristic, holding the others and industry fixed. This is the toy example from the slides (five stocks, one characteristic) with about 2,800 stocks and about 56 columns. Which style paid most in your month? Does that match what you remember?

In [ ]:
# The industries with the best and worst returns that month
ind = f[[c for c in f.index if c.startswith("ind_")]]
ind.index = [ind_names.get(int(c.split("_")[1]), c) for c in ind.index]
print("best :", (100*ind.nlargest(3)).round(1).to_dict())
print("worst:", (100*ind.nsmallest(3)).round(1).to_dict())

### 4.2 Run it for every month

This gives a time series of each factor's return, the same kind of object as Mkt-RF or HML, but built from a regression on many characteristics at once. We **leave Micron out** of the regression so that their own returns cannot move the factors that are then used to judge them.

In [ ]:
HOLD_OUT = panel.loc[panel.ticker == TICKER, "permno"].unique()   # the stock we study is left out

rows, r2s = {}, {}
for month, g in panel.groupby("month"):
    rows[month], r2s[month] = one_month(g[~g.permno.isin(HOLD_OUT)])
factors = pd.DataFrame(rows).T.sort_index()
print(factors.shape, "| average weighted R-squared:", round(np.mean(list(r2s.values())), 2))

# Check against the instructor's reference file (also estimated with MU held out)
common = factors.columns.intersection(ref_factors.columns)
diff = (factors[common] - ref_factors.loc[factors.index, common]).abs().max().max()
print("largest difference from the reference factor returns:", f"{diff:.2e}")

In [ ]:
# Cumulative return to each style factor, 2015-2026
(1 + factors[STYLES]).cumprod().plot(figsize=(8, 3.6), title="Cumulative return to style factors (cap-weighted WLS)")
plt.tight_layout(); plt.show()
print((100 * factors[STYLES].mean()).round(2).rename("average % per month").to_frame().T)

**Compare with the factors you know.** Does the momentum factor here behave like UMD? Check the correlation below. A style return and a Fama-French factor are different objects: one is a regression slope, the other a long-short portfolio. Note the sign on size: the exposure here is *ln(market cap)*, so large stocks score high, and its return moves opposite to SMB (small minus big).

In [ ]:
cmp = pd.DataFrame({"momentum (this model)": factors["momentum"], "UMD (French)": ff["umd"],
                    "value (this model)": factors["value"], "HML (French)": ff["hml"],
                    "size (this model)": factors["size"], "SMB (French)": ff["smb"]}).dropna()
cmp.corr().loc[["momentum (this model)", "value (this model)", "size (this model)"],
               ["UMD (French)", "HML (French)", "SMB (French)"]].round(2)

### 4.3 Attribution: where did the stock's return come from?

For one stock and month: **return = country + industry + (exposure x style return, summed over styles) + specific.** It adds up exactly.

In [ ]:
def attribution(permno, factors):
    d = panel[panel.permno == permno].set_index("month").sort_index()
    d = d[d.index.isin(factors.index)]
    out = pd.DataFrame(index=d.index)
    out["excess_return"] = d["y"]
    out["market"] = factors.loc[d.index, "country"]
    out["industry"] = [factors.loc[mo, f"ind_{int(i)}"] for mo, i in zip(d.index, d["industry"])]
    for s in STYLES:
        out[s] = d[s] * factors.loc[d.index, s]
    parts = ["market", "industry"] + STYLES
    out["specific"] = out["excess_return"] - out[parts].sum(axis=1)       # whatever the factors did not explain
    return out

att = attribution(permno, factors).loc[START:END]
parts = ["market", "industry"] + STYLES + ["specific"]
print("industry:", ind_names[int(panel.loc[panel.permno == permno, "industry"].iloc[-1])])
print((100 * att[["excess_return"] + parts].mean()).round(2).rename("average % per month").to_frame().T.to_string())

In [ ]:
# Same numbers as a chart: average monthly contribution of each source
avg = 100 * att[parts].mean()
ax = avg.plot.bar(figsize=(8, 3.2), color=["#0E6B63" if v >= 0 else "#B24E17" for v in avg])
ax.set_ylabel("% per month"); ax.set_title(f"{TICKER}: average monthly return by source, {START} to {END}")
plt.tight_layout(); plt.show()

**Your turn.** (1) How much of the average return is the market and the industry, before any stock-specific story? (2) Which style contributed most, and is that the style you expected? (3) Is the specific return statistically different from zero? Run the next cell.

In [ ]:
sp_mean, sp_t = nw_mean(att["specific"])
print(f"{TICKER}: average specific return {100*sp_mean:.2f}% per month, Newey-West t = {sp_t:.2f}")

### 4.4 Why hold the stock out?

If the stock is included in the regression, it helps determine its own industry factor, which pulls its specific return toward zero. Compare.

In [ ]:
rows_in = {mo: one_month(g)[0] for mo, g in panel.groupby("month")}          # nobody held out
att_in = attribution(permno, pd.DataFrame(rows_in).T.sort_index()).loc[START:END]
m_in, t_in = nw_mean(att_in["specific"])
print(f"held out : {100*sp_mean:.2f}% per month (t = {sp_t:.2f})")
print(f"included : {100*m_in:.2f}% per month (t = {t_in:.2f})")

## Part 5. Compare the three lenses (in class)

In [ ]:
def three_lenses(tk):
    pm = panel.loc[panel.ticker == tk, "permno"].mode().iloc[0]
    dd = dgtw[dgtw.permno == pm].set_index("month").loc[START:END]
    yy = panel[panel.permno == pm].set_index("month")["y"].loc[START:END]
    reg = sm.OLS(yy, sm.add_constant(ff.loc[yy.index, ["mktrf", "smb", "hml", "umd"]])).fit(cov_type="HAC", cov_kwds={"maxlags": 3})
    sp = attribution(pm, factors).loc[START:END]["specific"]
    (a, ta), (b, tb) = nw_mean(dd["dgtw_xret"]), nw_mean(sp)
    return {"ticker": tk, "months": len(yy),
            "DGTW xret %/mo": 100*a, "t (DGTW)": ta,
            "Carhart alpha %/mo": 100*reg.params["const"], "t (Carhart)": reg.tvalues["const"],
            "Barra specific %/mo": 100*b, "t (Barra)": tb}

pd.DataFrame([three_lenses(TICKER)]).set_index("ticker").round(2).T

**Discuss in groups (10 min).**

1. All three say "alpha" and all three are positive. Why are the numbers not the same? For each lens, say what it *controls for* and what it leaves in.
2. DGTW controls for three characteristics. The Carhart regression estimates loadings on four factors. The fundamental model also controls for **industry**. Micron is a memory-chip maker, and semiconductor stocks as a group did very well. What does that do to its specific return?
3. We held Micron out of the factor model. If you study a stock that was *in* the regression, which direction is the bias? (Part 4.4 shows it.)
4. We are studying Micron after seeing that it did very well, so the sample is not random. What does that do to the t-statistics above? What would you need to do to claim skill rather than luck? (This is where Session 4 starts.)
5. Which lens would you use to (a) report a fund's alpha, (b) judge a stock pick against look-alikes, (c) explain a portfolio's risk to a client?

**Optional.** Set `TICKER` to another stock and rerun Parts 2-5, adding it to `HOLD_OUT` first. Try one that did badly, and a large stable company. For a fair comparison, compare many stocks, not one.